# Train the Hangman Transformer on Kaggle GPU

Clones `approach/transformer` and trains there -- a char-level Transformer
encoder (BERT-style masked-language-model objective, trained from scratch
on train.txt, no pretrained checkpoint) with the same guessed-wrong-
letters/remaining-guesses features used on the BiLSTM branches.

Unlike the BiLSTM family, this has no recurrence at all -- every position
attends directly to every other position from layer one via self-
attention, with a learned positional embedding standing in for the
sequence-order information recurrence gave for free.

Uses a lower peak learning rate (3e-4 vs the BiLSTM branches' 1e-3) with
linear warmup before cosine decay, and higher dropout (0.2) -- Transformers
are more sensitive to LR early in training and more overfitting-prone on
a modest word list than recurrent models.

**Before running:** in the notebook's Settings panel (right sidebar), set
**Accelerator = GPU T4 x2** (or any GPU) and **Internet = On** (needed to `git clone`).

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected -- check Settings > Accelerator in the sidebar')

In [ ]:
REPO_URL = "https://github.com/Sahoo-Achyutananda/MELTWATER---HACKATHON.git"
BRANCH = "approach/transformer"

!rm -rf repo
!git clone --branch $BRANCH --single-branch $REPO_URL repo
%cd repo/brand-buzzword-hackathon
!ls

## Use the official competition dataset

The cloned repo carries its own copy of train.txt/test.txt (downloaded from
this same competition earlier), but overwrite them here so this notebook
verifiably sources data straight from Kaggle's own `/kaggle/input/`, not an
external GitHub copy -- same content, no ambiguity for anyone reviewing it.

In [ ]:
import shutil
shutil.copy("/kaggle/input/competitions/brand-buzzword-hackathon/train.txt", "train.txt")
shutil.copy("/kaggle/input/competitions/brand-buzzword-hackathon/test.txt", "test.txt")
print("train.txt and test.txt overwritten with the official competition dataset from /kaggle/input/")

## Train

Same masked-language-model objective + synthetic guessed-wrong/remaining
features as the BiLSTM branches. Bump `--epochs` up if you want -- warmup
and decay both automatically stretch to match whatever you pass.

In [ ]:
!python src/train_transformer.py --epochs 20

## Validate

Same held-out-train.txt methodology as every other branch. Compare
directly against BiLSTM+attention+features (approach/bilstm-attention-
features) to see whether pure attention (no recurrence at all) beats the
hybrid recurrent+attention approach.

In [ ]:
!python src/validate_transformer.py

## Generate submission.csv

Plays the actual game against every word in test.txt using the model
still in this session. Sandbox leaderboard checkpoint only -- per the
competition's Final Judgement policy, final hiring decisions re-run the
submitted model/notebook against a separate private word list.

250,000 words, one game at a time -- prints progress every 20,000 words
with an ETA.

In [ ]:
!python src/generate_submission_transformer.py

## Save outputs

Anything under `/kaggle/working/` is downloadable from the notebook's
Output tab after the run finishes.

In [ ]:
import shutil
shutil.copy("src/transformer_masker.pt", "/kaggle/working/transformer_masker.pt")
shutil.copy("submission.csv", "/kaggle/working/submission.csv")
print("saved transformer_masker.pt and submission.csv to /kaggle/working/ -- download from the Output tab")